In [10]:
from sklearn.model_selection import (
    cross_validate,
    RepeatedStratifiedKFold,
    train_test_split,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import pandas as pd

In [2]:
path = "assets/credit_card_fraud.csv"

In [3]:
df = pd.read_csv(path)

In [4]:
df.dropna(inplace=True)
df.isnull().sum()

CustomerID    0
A1            0
A2            0
A3            0
A4            0
A5            0
A6            0
A7            0
A8            0
A9            0
A10           0
A11           0
A12           0
A13           0
A14           0
Class         0
dtype: int64

In [8]:
X = df.drop("Class", axis=1)

In [9]:
y = df["Class"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVC": SVC(kernel="rbf", probability=True),
}

In [13]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scoring = ["accuracy", "f1_weighted", "roc_auc_ovr_weighted"]

In [14]:
results = {}
for name, model in models.items():
    cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    results[name] = {
        metric: f"{cv_results[f'test_{metric}'].mean():.4f} + {cv_results[f'test_{metric}'].std():.4f}"
        for metric in scoring
    }

/home/mano/Manoj/Learning/ml/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/mano/Manoj/Learning/ml/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/mano/Manoj/Learning/ml/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: Fut

In [15]:
benchmark_df = pd.DataFrame(results).T
print(benchmark_df)

                            accuracy      f1_weighted roc_auc_ovr_weighted
Logistic Regression  0.7923 + 0.0437  0.7905 + 0.0440      0.8731 + 0.0394
Random Forest        0.8659 + 0.0375  0.8658 + 0.0377      0.9384 + 0.0220
Gradient Boosting    0.8677 + 0.0269  0.8677 + 0.0271      0.9379 + 0.0192
SVC                  0.5362 + 0.0029  0.3744 + 0.0033      0.4683 + 0.0486
